In [ ]:
!pip -q uninstall -y scipy numpy llmcompressor transformers tokenizers accelerate datasets huggingface-hub safetensors compressed-tensors sentencepiece tqdm pandas
!pip -q cache purge

# torch는 test1처럼 cu128로 (대회 환경에 최대한 맞춤)
!pip -q install --no-cache-dir torch==2.9.0+cu128 --index-url https://download.pytorch.org/whl/cu128

# numpy + scipy를 같이 고정 (여기서 섞임 방지)
!pip -q install --no-cache-dir numpy==2.2.6 scipy==1.14.1

# 대회 기준 라이브러리 고정
!pip -q install --no-cache-dir \
  transformers==4.57.3 tokenizers==0.22.1 accelerate==1.10.1 datasets==4.4.1 huggingface-hub==0.36.0 \
  safetensors==0.7.0 compressed-tensors==0.13.0 sentencepiece==0.2.1 tqdm==4.67.1 pandas==2.3.3

# llmcompressor
!pip -q install --no-cache-dir llmcompressor


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 251.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 258.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 255.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cufflinks 0.17.3 requires pandas>=0.19.2, which is not installed.
bokeh 3.7.3 requires pandas>=1.2, which is not installed.
libpysal 4.14.1 requires pandas>=2.1.0, which is not installed.
yfinance 0.2.66 requires pandas>=1.3.0, which is not installed.
momepy 0.11.0 requires pandas>=2.0, which is not installed.
momepy 0.11.0 requires tqdm>=4.65, which is not installed.
spaghetti 1.7.6 requires pandas!=1.5.0,>=1.4, which is not installed.
mapclassify 2.10.0 requires pandas>=2.1, which is not installed

In [ ]:
import numpy as np, scipy, transformers, torch

print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available(), "| gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

import llmcompressor
from llmcompressor.modifiers.awq import AWQModifier
from llmcompressor import oneshot
print("llmcompressor OK ")


numpy: 2.2.6
scipy: 1.14.1
transformers: 4.57.3
torch: 2.9.0+cu128
cuda: True | gpu: Tesla T4
llmcompressor OK ✅


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    trust_remote_code=True,
).eval().to("cuda")

print("Loaded:", next(model.parameters()).device, next(model.parameters()).dtype)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loaded: cuda:0 torch.float16


In [ ]:
from datasets import load_dataset

NUM_CALIBRATION_SAMPLES = 64
MAX_SEQUENCE_LENGTH = 256

ds = load_dataset("LGAI-EXAONE/MANTA-1M", split=f"train[:{NUM_CALIBRATION_SAMPLES}]")

def preprocess(ex):
    return {
        "text": tokenizer.apply_chat_template(
            ex["conversations"],
            add_generation_prompt=True,
            tokenize=False
        )
    }

ds = ds.map(preprocess)
ds = ds.remove_columns([c for c in ds.column_names if c != "text"])

print("samples:", len(ds))


samples: 64


In [ ]:
from llmcompressor.modifiers.awq import AWQModifier
from llmcompressor import oneshot

recipe = [
    AWQModifier(
        mappings=None,
        ignore=["lm_head"],
        config_groups={
            "g0": {
                "targets": ["Linear"],
                "weights": {
                    "num_bits": 4,
                    "type": "int",
                    "strategy": "group",
                    "group_size": 128,
                    "symmetric": False
                }
            }
        },
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("AWQ int4 done")


Tokenizing:   0%|          | 0/64 [00:00<?, ? examples/s]

2026-02-03T12:07:55.325487+0000 | reset | INFO - Compression lifecycle reset
2026-02-03T12:07:55.330025+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-03T12:07:55.464763+0000 | on_initialize | INFO - No AWQModifier.mappings provided, inferring from model...
2026-02-03T12:07:55.467617+0000 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.0.self_attn.v_proj for mapping AWQMapping(smooth_layer='re:.*v_proj$', balance_layers=['re:.*o_proj$']) because found incompatible balance layers
2026-02-03T12:07:55.469260+0000 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.1.self_attn.v_proj for mapping AWQMapping(smooth_layer='re:.*v_proj$', balance_layers=['re:.*o_proj$']) because found incompatible balance layers
2026-02-03T12:07:55.470302+0000 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.2.self_attn.v_proj for mapping AWQMapping(smooth_layer='re:.*v_proj$', balance_layers=['re:.*o_proj$']) because found incompatible

(31/31): Calibrating: 100%|██████████| 64/64 [00:00<00:00, 1048.45it/s]
Smoothing: 0it [00:00, ?it/s]
(31/31): Propagating: 100%|██████████| 64/64 [00:00<00:00, 1113.68it/s]
Smoothing: 0it [00:00, ?it/s]
Calibrating weights: 210it [00:02, 85.31it/s]

2026-02-03T12:08:44.849357+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers


2026-02-03T12:08:44.906395+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
AWQ int4 done ✅


In [ ]:
import os, shutil, zipfile

OUT_DIR = "model"
os.makedirs(OUT_DIR, exist_ok=True)

# 모델 저장
model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

# 기존 test2.zip 있으면 삭제
if os.path.exists("test2.zip"):
    os.remove("test2.zip")

# test2.zip 생성 (최상위에 model/만)
shutil.make_archive(
    base_name="test2",
    format="zip",
    root_dir=".",
    base_dir="model"
)

# 구조 검증
with zipfile.ZipFile("test2.zip", "r") as z:
    top = sorted(set([n.split("/")[0] for n in z.namelist() if n.strip()]))
    print("top-level:", top)
    assert top == ["model"], "zip 최상위에 model 폴더만 있어야 합니다"

print("test2.zip ready ")

2026-02-03T12:08:44.944532+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 210it [00:11, 18.83it/s]


top-level: ['model']
test2.zip ready ✅
